In [1]:
import osmnx as ox
import geopandas as gpd
from pathlib import Path

RAW_DIR = Path("../../data/raw/05_red_viaria")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Descargando red viaria para camiones en Huesca...")

G = ox.graph_from_place(
    "Huesca, Spain",
    network_type="drive",        # solo vías transitables por vehículo
    simplify=True,
    custom_filter='["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link"]'
)

nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

print(f"Nodos:  {len(nodes):,}")
print(f"Tramos: {len(edges):,}")

output_gpkg = RAW_DIR / "carreteras_camiones_huesca.gpkg"
edges.to_file(output_gpkg, layer="edges", driver="GPKG")
nodes.to_file(output_gpkg, layer="nodes", driver="GPKG")
print(f"Guardado: {output_gpkg}")

print("\nTipos de vía:")
print(edges["highway"].explode().value_counts().to_string())

Descargando red viaria para camiones en Huesca...
Nodos:  4,881
Tramos: 9,373
Guardado: ..\..\data\raw\05_red_viaria\carreteras_camiones_huesca.gpkg

Tipos de vía:
highway
tertiary          2325
trunk             1945
secondary         1677
primary           1380
trunk_link         470
tertiary_link      421
secondary_link     373
primary_link       329
motorway_link      291
motorway           198


In [1]:
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pyproj import Transformer
from shapely.geometry import mapping
from PIL import Image
import io, base64
from pathlib import Path

RAW_DIR = Path("../../data/raw/05_red_viaria")
MAP_DIR = Path("../../docs/map/05_red_viaria")
SLOPE_FILE = Path("../../data/raw/03_pendiente_dem/pendiente_huesca_provincia.tif")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────
# 1. Pendiente clasificada como IMAGEN nítida (no heatmap)
#    Mismos 5 tramos/colores que el mapa 3D de pendiente
# ─────────────────────────────────────────────────────────
print("Leyendo pendiente...")
with rasterio.open(SLOPE_FILE) as src:
    slope = src.read(1).astype("float32")
    nodata = src.nodata or -9999
    slope[slope == nodata] = np.nan
    transform = src.transform
    width, height = src.width, src.height

# Submuestreo moderado solo por rendimiento del PNG (mucho más fino que los
# puntos de antes, sigue siendo nítido; sube/baja `step_img` según necesites)
step_img = 4
slope_img = slope[::step_img, ::step_img]

# --- Clasificación por color (idéntica al mapa 3D) ---
bins   = [5, 15, 30, 45]
colors_rgba = np.array([
    [34,  139, 34,  200],   # Plano
    [144, 238, 144, 200],   # Suave
    [220, 220, 50,  200],   # Moderado
    [255, 140, 0,   200],   # Fuerte
    [200, 30,  0,   200],   # Muy fuerte
], dtype=np.uint8)

idx = np.digitize(slope_img, bins)              # 0..4 según el tramo
rgba_img = colors_rgba[idx]                      # (H, W, 4)
rgba_img[np.isnan(slope_img)] = [0, 0, 0, 0]     # transparente donde no hay dato

png_img = Image.fromarray(rgba_img, mode="RGBA")
buf = io.BytesIO()
png_img.save(buf, format="PNG")
png_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
image_data_uri = f"data:image/png;base64,{png_b64}"

# --- Bounds de la imagen en WGS84 (esquinas del raster reproyectadas) ---
tr = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
corners_x = [transform.c, transform.c + width * transform.a]
corners_y = [transform.f, transform.f + height * transform.e]
lon_corners, lat_corners = tr.transform(
    [corners_x[0], corners_x[1], corners_x[1], corners_x[0]],
    [corners_y[0], corners_y[0], corners_y[1], corners_y[1]],
)
west, east = min(lon_corners), max(lon_corners)
south, north = min(lat_corners), max(lat_corners)

print(f"Imagen de pendiente: {rgba_img.shape[1]}x{rgba_img.shape[0]} px")

# ─────────────────────────────────────────────────────────
# 2. Carreteras — paleta clara y con contraste real
# ─────────────────────────────────────────────────────────
print("Leyendo carreteras...")
edges = gpd.read_file(RAW_DIR / "carreteras_camiones_huesca.gpkg", layer="edges").to_crs("EPSG:4326")

# Paleta clara/luminosa: se lee bien tanto sobre el fondo oscuro del basemap
# como sobre el verde/naranja/rojo de la capa de pendiente
color_map = {
    "motorway":  [255, 255, 255, 255],   # blanco   — máxima jerarquía
    "trunk":     [120, 200, 255, 255],   # celeste
    "primary":   [255, 221, 120, 255],   # amarillo suave
    "secondary": [210, 210, 210, 230],   # gris claro
    "tertiary":  [150, 150, 150, 210],   # gris medio
}
DEFAULT_COLOR = [150, 150, 150, 210]

def get_color(hw):
    if isinstance(hw, list): hw = hw[0]
    for key, val in color_map.items():
        if isinstance(hw, str) and key in hw:
            return val
    return DEFAULT_COLOR

def get_width(hw):
    if isinstance(hw, list): hw = hw[0]
    if isinstance(hw, str):
        if "motorway" in hw or "trunk" in hw: return 150
        if "primary" in hw:                   return 110
        if "secondary" in hw:                 return 75
    return 45

edges["color"] = edges["highway"].apply(get_color)
edges["width"] = edges["highway"].apply(get_width)

features = []
for _, row in edges.iterrows():
    hw = row["highway"]
    if isinstance(hw, list): hw = hw[0]
    tipo = {
        "motorway":  "Autopista",
        "trunk":     "Vía rápida",
        "primary":   "Carretera nacional",
        "secondary": "Carretera comarcal",
        "tertiary":  "Carretera local",
    }.get(str(hw).split("_")[0], str(hw))

    features.append({
        "type": "Feature",
        "geometry": mapping(row.geometry),
        "properties": {
            "color":      row["color"],
            "width":      row["width"],
            "width_halo": row["width"] + 60,
            "tipo_via":   tipo,
            "nombre":     str(row.get("name", "—") or "—"),
        }
    })
geojson_roads = {"type": "FeatureCollection", "features": features}

# ─────────────────────────────────────────────────────────
# 3. Capas
# ─────────────────────────────────────────────────────────

# Pendiente: imagen nítida clasificada (reemplaza al HeatmapLayer)
# IMPORTANTE: pydeck antepone "@@=" a cualquier string pasado como argumento,
# asumiendo que es un accessor (columna de datos) a evaluar en JS. "image" es
# un valor literal, no un accessor -- hay que envolverlo en pdk.types.String
# para que pydeck lo serialice tal cual, sin el prefijo que rompe el render.
layer_slope = pdk.Layer(
    "BitmapLayer",
    image=pdk.types.String(image_data_uri),
    bounds=[west, south, east, north],
    opacity=0.85,
)

# Halo negro debajo de las carreteras (da contraste y legibilidad)
layer_roads_halo = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_roads,
    stroked=True,
    filled=False,
    get_line_color=[0, 0, 0, 220],
    get_line_width="properties.width_halo",
    line_width_min_pixels=4,
    pickable=False,
)

# Trazo real de carreteras encima
layer_roads = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_roads,
    stroked=True,
    filled=False,
    get_line_color="properties.color",
    get_line_width="properties.width",
    line_width_min_pixels=2,
    pickable=True,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=7,
    max_zoom=13,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": "<b>{tipo_via}</b><br/>{nombre}",
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

deck = pdk.Deck(
    layers=[layer_slope, layer_roads_halo, layer_roads],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

# NOTA: position:fixed (no absolute) — así la leyenda se ancla siempre al
# viewport completo y no se corta, igual que en el mapa 3D de pendiente.
leyenda_html = """
<div style="position:fixed; bottom:30px; left:20px; background:rgba(0,0,0,0.75);
            padding:14px 18px; border-radius:8px; font-family:sans-serif;
            font-size:13px; color:white; z-index:9999; line-height:1.9;
            box-shadow:0 2px 8px rgba(0,0,0,0.5);">

  <div style="font-weight:bold; font-size:14px; margin-bottom:8px;">🗺 Leyenda</div>

  <div style="margin-bottom:6px; font-weight:bold; color:#aaa;">Pendiente del terreno</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#228B22;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>0–5° Plano</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#90EE90;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>5–15° Suave</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#DCDC32;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>15–30° Moderado</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#FF8C00;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>30–45° Fuerte</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#C81E00;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>&gt;45° Muy fuerte</div>

  <div style="margin:10px 0 6px; font-weight:bold; color:#aaa;">Red viaria</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#ffffff;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Autopista</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#78c8ff;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Vía rápida</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#ffdd78;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera nacional</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#d2d2d2;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera comarcal</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#969696;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera local</div>
</div>
"""

html_output = deck.to_html(as_string=True)
html_output = html_output.replace("</body>", leyenda_html + "</body>")

with open(MAP_DIR / "mapa_pendiente_carreteras_huesca.html", "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"Guardado: {MAP_DIR / 'mapa_pendiente_carreteras_huesca.html'}")


Leyendo pendiente...
Imagen de pendiente: 1411x1730 px
Leyendo carreteras...
Guardado: ..\..\docs\map\05_red_viaria\mapa_pendiente_carreteras_huesca.html
